In [2]:
# --- Make the shared `techintel` package importable ---------------------------
# The WoS parsing / graph / metrics helpers used to be copy-pasted into each
# notebook. They now live in techintel/ at the repository root; this finds that
# root by walking up from the working directory.
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "techintel" / "__init__.py").exists():
            return candidate
    raise RuntimeError(
        "Could not find the repository root (a directory containing 'techintel/')."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from techintel import communities, graphs, metrics

# Point this at your authorship GraphML file. (The original hard-coded an
# absolute /Users/... path, which only worked on one machine.)
GRAPHML_FILE = "giant_component_authorship_network.graphml"


def analyze_authorship_communities(graphml_file):
    """Louvain communities over the graph, then a macro-analysis row per community.

    `graphs.load_graph` coerces to undirected and strips self-loops, and
    `metrics.macro_analysis` supplies the v/e/p/u counts, alpha/beta/gamma,
    degree stats, triangles, transitivity and assortativity.
    """
    G = graphs.load_graph(graphml_file, undirected=True, remove_self_loops=True)

    print("Graph loaded")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())

    if G.number_of_edges() == 0:
        print("No edges in the graph. Cannot compute modularity.")
        return None

    partition = communities.louvain_partition(G)
    modularity = communities.partition_modularity(G, partition)
    print(f"\nModularity: {modularity}")
    print(f"Communities detected: {len(set(partition.values()))}")

    communities.label_communities(G, partition)

    rows = []
    for community_id in sorted(set(partition.values())):
        nodes_in_community = [n for n, c in partition.items() if c == community_id]
        metrics_row = metrics.macro_analysis(G.subgraph(nodes_in_community).copy())
        metrics_row["Community"] = community_id
        rows.append(metrics_row)

    df = pd.DataFrame(rows)
    print("\nCommunity Macro Analysis:\n")
    print(df.to_string(index=False))
    return df


df = analyze_authorship_communities(GRAPHML_FILE)


✅ Graph loaded
Nodes: 164274
Edges: 928045

Modularity: 0.751602085642659
Communities detected: 148

Community Macro Analysis:

 Nodes  Edges    Alpha     Beta    Gamma  Avg Degree  Heterogeneity  Triangles  Transitivity  Isolated Nodes  Assortativity  Community
  3061  12658 1.569070 4.135250 0.002703    8.270500     132.013730    26757.0      0.839130               0      -0.031687          0
  1763   6872 1.451292 3.897901 0.004424    7.795803      90.506233    13831.0      0.866050               0      -0.056881          1
  2768  10998 1.488158 3.973266 0.002872    7.946532      65.068673    23391.0      0.856566               0      -0.021757          2
  2295   9299 1.527808 4.051852 0.003533    8.103704      74.386631    20639.0      0.844803               0       0.001965          3
  1504   6230 1.574093 4.142287 0.005512    8.284574      79.466890    13889.0      0.856680               0       0.050987          4
  4018  17759 1.711119 4.419861 0.002201    8.839721     155.5